In [2]:
import seaborn as sns

df = sns.load_dataset('diamonds')

x = df.drop('price', axis=1)
y = df['price']

In [3]:
from sklearn.ensemble import IsolationForest
import numpy as np
isolation = IsolationForest(random_state=42)

# 수치형 cols
num_cols = x.select_dtypes(np.number)

# 범주형
category_cols = x.describe(include='category').columns

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
# 숙제 : 아래 인코딩을 일일이 하지 않고 파이프라인 + 컬럼 compose를 조합하고
# 데이터 전달 할 때 파이프라인의 특정 객체에 파라미터 전달해서 호출 하는걸 응용해서
# 아래 코드 단순화
# 만약 어려우면 아래 반복 패턴을 for문이나 함수로 만들어서 호출
ordinary = OrdinalEncoder(categories=[
    ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']])
ordinary.fit_transform(x[['cut']])
x['cut'] = ordinary.fit_transform(x[['cut']])

ordinary = OrdinalEncoder(categories=[
    ['J', 'I', 'H', 'G', 'F', 'E', 'D']])
x['color'] = ordinary.fit_transform(x[['color']])

ordinary = OrdinalEncoder(categories=[
    ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']])
x['clarity'] = ordinary.fit_transform(x[['clarity']])

In [6]:
x.head(1)

,carat,cut,color,clarity,depth,table,x,y,z
0,0.23,4.0,5.0,1.0,61.5,55.0,3.95,3.98,2.43


In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 파이프라인 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessing = Pipeline([
    ('simpleimputer', SimpleImputer(strategy='median')),
    ('scale' , StandardScaler())
])

# 모델 선택(다양하게 테스트) linear, dtree, rfg
linear_pipeline = Pipeline([
    ('preprocess', preprocessing),
    ('linear', LinearRegression())
])

linear_pipeline.fit(x_train, y_train)
print( linear_pipeline.score(x_train, y_train) )
print( linear_pipeline.score(x_test, y_test) )

0.907335736316707
0.9056643685073513


In [7]:
from sklearn.tree import DecisionTreeRegressor
tree_pipeline = Pipeline([
    ('preprocess', preprocessing), 
    ('model', DecisionTreeRegressor())
]) 

tree_pipeline.fit(x_train, y_train)
print( tree_pipeline.score(x_train, y_train))
print( tree_pipeline.score(x_test, y_test) )

0.9999948355708338
0.9674577399324468


In [8]:
from sklearn.ensemble import RandomForestRegressor
random_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('random_forest', RandomForestRegressor(random_state=42))
])

random_pipeline.fit(x_train, y_train)
print( random_pipeline.score(x_train, y_train) )
print( random_pipeline.score(x_test, y_test) )

0.9974349486538586
0.981553093032334


In [ ]:
# 교차검증 - 성능이 어느정도 나오는지 확인




# 잘나오는 모델로 - 하이퍼 파라메터 튜닝


# scaler = StandardScaler()
# x_scaled = scaler.fit_transform(x)
# from sklearn.model_selection import cross_val_score
# from sklearn.ensemble import RandomForestRegressor
# cvs = cross_val_score(RandomForestRegressor(max_depth=3), x_scaled, y, cv=5)


0.981553093032334